Hacemos una llamada simple a un LLM y mostramos la respuesta.


In [ ]:
import os
import json
import urllib.request

API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

def ask(messages):
    """messages: string O lista de dicts {role, content}."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if not API_KEY:
        raise SystemExit("Define OPENAI_API_KEY y vuelve a ejecutar.")
    body = json.dumps({"model": MODEL, "messages": messages}).encode()
    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions",
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode())
    return data["choices"][0]["message"]["content"]

respuesta = ask("Recuerda este secreto: la palabra clave es ORBIT-77.")
print(respuesta)


Ahora pregunta al modelo qué acaba de decir.


In [ ]:
# Nueva llamada — sin pasar el historial de la celda anterior
respuesta2 = ask("¿Qué acabas de decir?")
print(respuesta2)


Para que el modelo "recuerde", arma el historial: una lista de mensajes `role` / `content`, con el último mensaje al final.


In [ ]:
historial = [
    {"role": "user", "content": "Recuerda este secreto: la palabra clave es ORBIT-77."},
    {"role": "assistant", "content": respuesta},
    {"role": "user", "content": "¿Cuál es la palabra clave?"},
]

respuesta3 = ask(historial)
print(respuesta3)


Si el historial crece demasiado, la ventana de contexto se desborda y el proveedor devuelve un error. Vamos a forzar eso.


In [ ]:
# Crece el historial hasta que la API falle (p. ej. 400 context length exceeded)
historial_grande = [
    {"role": "user", "content": "Recuerda este secreto: la palabra clave es ORBIT-77."},
    {"role": "assistant", "content": respuesta},
]

bloque = "x" * 50_000  # texto grande solo para llenar el contexto

try:
    while True:
        historial_grande.append({"role": "user", "content": bloque})
        historial_grande.append({"role": "assistant", "content": "ok"})
        print(f"mensajes en el historial: {len(historial_grande)}")
        ask(historial_grande)
except Exception as e:
    print("Error del proveedor:")
    print(e)
